In [1]:
import os
import openai
import json
from tqdm.notebook import tqdm
import numpy as np
from openai import AsyncOpenAI
import re
from PIL import Image
from pprint import pprint
import warnings

%matplotlib inline
warnings.filterwarnings('ignore', category=UserWarning, module='pandas')

# initialize openai
from dotenv import load_dotenv
load_dotenv()
# openai.api_key = os.environ["OPENAI_API_KEY"]

ModuleNotFoundError: No module named 'dotenv'

- 이미지 cosine similarity
- 이미지 description
	- intro to meta data
		- furniture의 개수 등을 기준으로 filter
- 사진들간의 cosine similarity
	- types of furniture

### 필요한 데이터 load

- `img_path` : 이미지 path
- `img_descriptions` : GPT-4V가 생성한 이미지에 대한 설명
- `furniture_paths` : YOLO가 detect한 가구들의 이미지 path
- `furniture_descriptions` : YOLO가 detect한 가구들

In [ ]:
img_paths = list(os.walk('03.Image_embedding_data/room-dataset/living'))[0][2]
img_paths = [i for i in img_paths if i!=".DS_Store"]
img_paths = [i for i in img_paths if int(i.split('_')[1].split('.')[0]) in list(range(1, 101))]

def extract_number(filename):
    match = re.search(r'\d+', filename)
    return int(match.group()) if match else 0

img_paths = sorted(img_paths, key=extract_number)
img_paths = [os.path.join('03.Image_embedding_data/room-dataset/living', i) for i in img_paths]

img_paths[:10]

In [ ]:
with open("03.Image_embedding_data/room-dataset/room_descriptions_parsed.json", 'r') as file:
    img_descriptions = json.load(file)

img_descriptions['living_100.jpg']

In [ ]:
furniture_paths = list(os.walk('03.Image_embedding_data/room-dataset/living_cropped'))[0][2]
furniture_paths = [i for i in furniture_paths if i!=".DS_Store"]
furniture_paths = [os.path.join('03.Image_embedding_data/room-dataset/living_cropped', i) for i in furniture_paths]

furniture_paths[:10]

In [ ]:
with open("03.Image_embedding_data/room-dataset/room_detections_parsed.json", 'r') as file:
    furniture_descriptions = json.load(file)

furniture_descriptions['03.Image_embedding_data/room-dataset/living/living_1.jpg']

In [ ]:
Image.open(img_paths[10])

In [ ]:
pprint(img_descriptions['living_11.jpg'])

In [ ]:
furniture_descriptions['03.Image_embedding_data/room-dataset/living/living_11.jpg']

In [ ]:
from utils import draw_images

draw_images([Image.open(i) for i in furniture_paths if 'living_11' in i])

---

### 1. 이미지 전체의 cosine similarity를 활용한 유사한 이미지 탐색

#### < DB 구축 >
- CLIP을 활용하여 이미지들을 Image embedding vector로 변환 (01.Image search.ipynb 참고)

#### < Search >
- Input image도 마찬가지로 Image embedding vector로 변환
- DB와 cosine similarity 유사도를 기반으로 search

1-1. DB 구축

In [ ]:
from transformers import CLIPProcessor, CLIPModel
from utils import extract_img_features, search_image

# https://huggingface.co/

model_name = "openai/clip-vit-base-patch32"
clip_model = CLIPModel.from_pretrained(model_name)
clip_processor = CLIPProcessor.from_pretrained(model_name)

In [ ]:
image_features = [extract_img_features(Image.open(i), clip_processor, clip_model) for i in img_paths]

In [ ]:
import pandas as pd

In [ ]:
df = pd.DataFrame({"img_path":img_paths, "img_emb":image_features})
df['img_name'] = df['img_path'].str.split('/').str[-1]
df.head()

In [ ]:
query_image = Image.open("03.Image_embedding_data/room-dataset/living/living_212.jpg")
query_image

1-2. Search

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

def search_similar_vector(query_feature, features):
    """
    주어진 vector들과 비교하여, query_feature와 유사한 vector의 index와 유사도를 제공함

    Args:
        query_feature (np.array): input embedding vector
        features (List[np.array]): embedding vector들의 list

    Returns:
        Tuple[np.array, np.array]: 유사한 embedding vector들의 index & cosine similarity
    """
    features_stack = np.vstack(features)
    similarities = cosine_similarity([query_feature], features_stack).flatten()

    return similarities

In [ ]:
df['clip_similarities'] = search_similar_vector(extract_img_features(query_image, clip_processor, clip_model)[0], df.img_emb.tolist())

In [ ]:
df.head()

In [ ]:
top5 = df.nlargest(5, 'clip_similarities')

In [ ]:
top5

In [ ]:
draw_images([Image.open(i) for i in top5.img_path.tolist()], top5.img_name.tolist())

---

### 2. 이미지의 description을 활용한 유사도 탐색

#### < Text embedding db 추가 >
- 02.알맞은 embedding model 선택 방법 참고
- 04.유사도 측정 최적화 참고

#### < Input query 전처리 >
- GPT-4V를 활용하여 동일한 형태의 image description 생성
- Text embedding화 하여 기존의 text embedding db와 유사도 측정

1-1. Text embedding DB 구축

In [ ]:
df.head(2)

In [ ]:
img_descriptions['living_1.jpg']

In [ ]:
img_desc_df = pd.DataFrame(img_descriptions).T.reset_index()
img_desc_df.rename(columns={"index":"img_name"}, inplace=True)

In [ ]:
img_desc_df.head(2)

In [ ]:
df = df.merge(img_desc_df, on='img_name', how='left')

In [ ]:
df.isna().sum()

In [ ]:
df.head(2)

In [ ]:
from text_utils import create_embeddings

In [ ]:
# embedding api를 활용하여 batch로 처리 가능
df['Color Scheme emb'] = create_embeddings(df['Color Scheme'].tolist())
df['Lighting emb'] = create_embeddings(df['Lighting'].tolist())
df['Spatial Layout emb'] = create_embeddings(df['Spatial Layout'].tolist())
df['Architectural Features emb'] = create_embeddings(df['Architectural Features'].tolist())

In [ ]:
df.head(2)

1-2. Input query 전처리 및 search

- input query 전처리

In [ ]:
from text_utils import normal_chat_completion

In [ ]:
prompt = """Reformat the input Korean, into a json format like below. The output should be in English.
It should capture information related to 'Color Scheme', 'Lighting', 'Spatial Layout', 'Architectural Features'.
If there is no information related to each category, create one yourself.

Example output:
{'Color Scheme': 'The living room features a neutral color palette with earthy tones. White walls dominate the space, providing a bright and open feel, while furniture pieces in shades of beige, cream, and brown add warmth. Accents of dark wood on the ceiling beams and furniture give the room a rich contrast, and subtle patterns on the upholstery create visual interest without overpowering the space.',
 'Lighting': 'Natural light streams in through the arched windows, illuminating the room and highlighting the indoor greenery. The ceiling pendant and strategically placed floor and table lamps provide additional layers of warm ambient lighting that contribute to the cozy atmosphere.',
 'Spatial Layout': 'The layout is open and inviting, with a central seating area comprised of a large sofa and complementing armchairs facing each other over a leather ottoman that doubles as a coffee table. The arrangement encourages conversation and social interaction. Clear pathways around the furniture make for easy movement through the room.',
 'Architectural Features': "Architectural highlights include the high ceiling adorned with dark wooden beams that add character and a sense of history to the space. The ceiling's design, along with the arched windows and doorways, suggests a Mediterranean or Spanish influence. These features are complemented by a mix of classic and contemporary furniture, creating a timeless look."}

Input Korean:
"""

room_desc = "벽난로가 가운데에 있고, 전체적인 분위기는 어두웠으면 좋겠어. 그리고 나무 탁자가 가운데에 배치되어 있어야 하고, 특색있는 소파들이 주변에 있으면 좋겠어. 두 개는 작은 소파고, 하나는 3인용 소파로. 그리고 주변에 포인트를 줄 수 있는 가구들이 있는 것도 좋아."

In [ ]:
output = normal_chat_completion(prompt + room_desc)

In [ ]:
image_desc = json.loads(output.choices[0].message.content)

In [ ]:
image_desc

In [ ]:
image_desc_emb = dict()

for k,v in image_desc.items():
    image_desc_emb[k] = create_embeddings(v)[0]

In [ ]:
image_desc_emb

In [ ]:
image_desc_emb.keys()

search

In [ ]:
df['color_sim'] = search_similar_vector(image_desc_emb['Color Scheme'], df['Color Scheme emb'].tolist())
df['lighting_sim'] = search_similar_vector(image_desc_emb['Lighting'], df['Lighting emb'].tolist())
df['layout_sim'] = search_similar_vector(image_desc_emb['Spatial Layout'], df['Spatial Layout emb'].tolist())
df['archi_sim'] = search_similar_vector(image_desc_emb['Architectural Features'], df['Architectural Features emb'].tolist())

In [ ]:
df['desc_similarity'] = df[['color_sim', 'lighting_sim', 'layout_sim', 'archi_sim']].mean(axis=1)

In [ ]:
df.head(2)

In [ ]:
img_top5 = df.nlargest(5, 'clip_similarities')
draw_images([Image.open(i) for i in img_top5['img_path'].tolist()], img_top5.img_name.tolist())

In [ ]:
text_top5 = df.nlargest(5, 'desc_similarity')
draw_images([Image.open(i) for i in text_top5['img_path'].tolist()], text_top5.img_name.tolist())

In [ ]:
text_top5 = df.nlargest(5, 'layout_sim')
draw_images([Image.open(i) for i in text_top5['img_path'].tolist()], text_top5.img_name.tolist())

In [ ]:
text_top5 = df.nlargest(5, 'archi_sim')
draw_images([Image.open(i) for i in text_top5['img_path'].tolist()], text_top5.img_name.tolist())

- 각 description 별로 원하는 대로 weight를 다르게 줄 수 있음 (similarity의 시각을 다각화)

==> Meta data를 활용, 기존의 search space를 줄일 수도 있음

In [ ]:
from collections import Counter

In [ ]:
furniture_descriptions['03.Image_embedding_data/room-dataset/living/living_1.jpg']

In [ ]:
filter_imgs = list()

for k,v in furniture_descriptions.items():
    counts = Counter(v['labels'])
    if (counts['couch']>=1) and (counts['dining table']>=1):
        filter_imgs.append(k)

In [ ]:
filter_imgs

In [ ]:
draw_images([Image.open(i) for i in filter_imgs])

In [ ]:
tmp_df = df.loc[df['img_path'].isin(filter_imgs)]

In [ ]:
tmp_df

In [ ]:
text_top5 = tmp_df.nlargest(5, 'desc_similarity')
draw_images([Image.open(i) for i in text_top5['img_path'].tolist()], text_top5.img_name.tolist())

---

### 3. user query에 보인 가구와 유사한 상품 탐색

#### < 가구 DB 구축 >
- crop된 이미지들의 embedding vector 생성

#### < 이미지에서 사용된 가구와 유사한 가구 search >
- Input 이미지의 분위기를 연출하기 위해 필요한 가구들

In [ ]:
furniture_paths[:5]

In [ ]:
furniture_df = pd.DataFrame(furniture_paths)
furniture_df.columns = ['path']
furniture_df['f_img_name'] = furniture_df['path'].str.split('/').str[-1]
furniture_df['original_img'] = furniture_df['f_img_name'].apply(lambda x: "_".join(x.split("_")[:2]))
furniture_df['furniture_id'] = furniture_df['f_img_name'].str.extract(r'_(\d+)\.jpg$')[0]

In [ ]:
furniture_df.head(2)

In [ ]:
furniture_df.sort_values(by=['original_img'])

In [ ]:
furniture_descriptions['03.Image_embedding_data/room-dataset/living/living_1.jpg']

In [ ]:
furniture_desc_df = pd.DataFrame()

for k,v in furniture_descriptions.items():
    tmp_df = pd.DataFrame(v)
    tmp_df['furniture_id'] = [str(i) for i in list(range(len(tmp_df)))]
    match = re.search(r'(living_\d+)', k)
    f_tmp_df = furniture_df.loc[furniture_df['original_img']==match.group()].sort_values(by='furniture_id')
    if len(f_tmp_df)>0:
        f_tmp_df = f_tmp_df.merge(tmp_df, on='furniture_id', how='left')
        furniture_desc_df = pd.concat([furniture_desc_df, f_tmp_df], axis=0)

In [ ]:
furniture_desc_df.head(2)

embedding vector 생성하기

In [ ]:
from transformers import CLIPProcessor, CLIPModel
from utils import extract_img_features

# https://huggingface.co/

model_name = "openai/clip-vit-base-patch32"
clip_model = CLIPModel.from_pretrained(model_name)
clip_processor = CLIPProcessor.from_pretrained(model_name)

In [ ]:
embeddings = [extract_img_features(Image.open(i), clip_processor, clip_model) for i in furniture_desc_df.path.tolist()]
furniture_desc_df['emb'] = embeddings

In [ ]:
furniture_desc_df.head(2)

가구 이미지 추출

In [ ]:
from utils import detect_objects, filter_furniture, crop_bbox, normalize_image

import yolov5

# 출처 : https://pypi.org/project/yolov5/

# load pretrained model
model = yolov5.load('yolov5s.pt')

# set model parameters
model.conf = 0.3  # NMS confidence threshold
model.iou = 0.45  # NMS IoU threshold
model.agnostic = False  # NMS class-agnostic
model.multi_label = False  # NMS multiple labels per box
model.max_det = 1000  # maximum number of detections per image

In [ ]:
detect = detect_objects('03.Image_embedding_data/room-dataset/living/living_212.jpg', model)
detections_parsed = filter_furniture(detect)

In [ ]:
detections_parsed

In [ ]:
detect[0].show()

In [ ]:
# detections_parsed
boxes = list()

for b in detections_parsed['boxes']:
    cropped = crop_bbox(query_image, b)
    normalized_image = normalize_image(cropped, target_size=(112, 112))
    boxes.append(normalized_image)

In [ ]:
%matplotlib inline
draw_images(boxes, detections_parsed['lables']) # dining table or chair?

현재 이미지에 있는 가구들과 유사한 가구들을 search

In [ ]:
boxes_emb = [extract_img_features(i, clip_processor, clip_model)[0] for i in boxes]

In [ ]:
search_df = furniture_desc_df[['path', 'f_img_name']]
search_df['table_findings'] = search_similar_vector(boxes_emb[4], furniture_desc_df['emb'].tolist())
search_df['chair_findings'] = search_similar_vector(boxes_emb[5], furniture_desc_df['emb'].tolist())

In [ ]:
search_df.head()

In [ ]:
table_top5 = search_df.nlargest(5, 'table_findings')
chair_top5 = search_df.nlargest(5, 'chair_findings')

In [ ]:
draw_images([Image.open(i) for i in table_top5['path']])

In [ ]:
draw_images([Image.open(i) for i in chair_top5['path']])

--END--